<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_6_Multi%E2%80%91Agent_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D1%8B_%D1%81%D0%BE%D0%B2%D0%BC%D0%B5%D1%81%D1%82%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D0%BE%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.6. Multi‑Agent системы: совместная работа агентов



## Введение: от одиночного агента к команде экспертов

В предыдущих лекциях мы построили мощного одиночного агента: он умеет искать в документах, вычислять, запоминать контекст, вызывать инструменты и даже принимать решения. Мы научили его быть «мастером на все руки». Однако для сложных задач — например, написание аналитического отчёта, исследование рынка или планирование проекта — один агент быстро достигает предела. Ему приходится одновременно выполнять роли исследователя, аналитика, писателя и критика, что перегружает его контекстное окно и снижает качество работы.

В реальной жизни мы редко делаем всё сами. Мы делегируем: один человек ищет информацию, другой проверяет факты, третий пишет текст, четвёртый редактирует. Такой подход называется **multi‑agent** (многоагентный): мы создаём несколько специализированных агентов, каждый из которых отвечает за свою задачу, и они обмениваются результатами, уточняя и дополняя друг друга.

В этой лекции мы построим систему из двух‑трёх агентов, работающих в цикле. Мы увидим, как разделение обязанностей повышает качество ответов, делает систему более прозрачной и модульной. В финале мы добавим реальные инструменты (поиск, калькулятор), чтобы агенты стали по-настоящему полезными.

---

## Тема 1. Зачем делить агентов, а не делать одного супер‑агента

Прежде чем мы начнём программировать, давайте разберёмся, почему multi‑agent подход лучше, чем попытка «впихнуть всё в одного».

### 1.1. Проблемы одного агента

Один универсальный агент сталкивается с рядом фундаментальных ограничений:

| Проблема | Описание | Пример |
|----------|----------|--------|
| **Перегрузка контекста** | Агент должен помнить и обрабатывать огромное количество информации | При написании отчёта агент держит в голове 50 документов, историю диалога и инструкции — контекстное окно переполняется |
| **Смешение ролей** | Один агент должен быть и исследователем, и аналитиком, и писателем — это снижает качество каждой роли | Модель «в режиме поиска» может не уметь хорошо писать, и наоборот |
| **Сложность отладки** | Если агент ошибся, непонятно, на каком этапе произошла ошибка | Агент нашёл не те документы, неправильно их проанализировал или плохо сформулировал вывод? |
| **Ограниченная специализация** | Один промпт не может идеально настроить модель на все задачи одновременно | Один и тот же агент не может быть одновременно строгим фактчекером и креативным писателем |
| **Трудность распараллеливания** | Агент выполняет шаги последовательно, хотя некоторые можно делать параллельно | Поиск информации и анализ можно делать одновременно |

### 1.2. Преимущества multi‑agent подхода

Разделение обязанностей между несколькими агентами даёт преимущества:

| Преимущество | Описание |
|--------------|----------|
| **Модульность** | Каждый агент решает одну задачу — проще разрабатывать, тестировать и заменять |
| **Параллелизм** | Агенты могут работать одновременно, ускоряя процесс |
| **Специализация** | Каждый агент использует свой промпт и свои инструменты, настроенные под его роль |
| **Прозрачность** | Видно, какой агент что сделал — легко отлаживать |
| **Гибкость** | Можно использовать разные модели для разных ролей (например, быструю модель для поиска, большую для анализа) |
| **Масштабируемость** | Легко добавить нового агента для новой роли |

**Пример:** для написания отчёта мы можем создать:
1. **Агент-исследователь** — ищет информацию в документах и интернете.
2. **Агент-аналитик** — проверяет факты и структурирует данные.
3. **Агент-писатель** — пишет текст на основе структурированных данных.
4. **Агент-критик** — проверяет готовый отчёт на ошибки.

Каждый агент делает свою работу лучше, чем один универсальный агент.

### 1.3. Способы взаимодействия агентов

Агенты могут общаться друг с другом разными способами:

| Паттерн | Описание | Схема |
|---------|----------|-------|
| **Последовательный (цепочка)** | Агенты выполняются один за другим, передавая результат следующему | Исследователь → Аналитик → Писатель → Критик |
| **Итеративный (диалог)** | Агенты обмениваются сообщениями в цикле, уточняя результат | Писатель → Критик → Писатель (исправляет) → Критик (проверяет) |
| **Иерархический (супервайзер)** | Один агент-координатор управляет другими, решая, кого и когда вызывать | Супервайзер → Исследователь → Супервайзер → Писатель → Супервайзер |
| **Групповое обсуждение** | Все агенты одновременно обсуждают проблему и приходят к общему решению | Агент1 ↔ Агент2 ↔ Агент3 |

В этой лекции мы реализуем **иерархический паттерн с супервайзером** — самый гибкий и мощный способ организации.

### 1.4. Обзор популярных паттернов multi‑agent систем

| Паттерн | Описание | Когда использовать |
|---------|----------|-------------------|
| **Докладчик‑критик** | Один агент пишет, другой критикует и предлагает улучшения | Для генерации контента |
| **Планировщик‑исполнитель** | Один агент составляет план, другой его выполняет | Для сложных задач |
| **Групповое обсуждение** | Несколько агентов обсуждают проблему и приходят к консенсусу | Для принятия решений |
| **Ролевая игра** | Каждый агент играет роль (эксперт, скептик, оптимист) | Для всестороннего анализа |
| **Супервайзер + работники** | Один агент управляет другими, делегируя задачи | Для любой сложной задачи |

Мы реализуем паттерн **«Супервайзер + работники»**, потому что он самый универсальный.

### 1.5. Установка необходимых пакетов

Для работы с multi‑agent системами нам понадобятся те же пакеты, что и раньше:

```bash
pip install langchain langchain-ollama langgraph chromadb sentence-transformers duckduckgo-search
```

Если вы прошли предыдущие лекции, всё уже установлено. `duckduckgo-search` добавим для веб-поиска, который понадобится в следующих темах.

---

В следующей части мы перейдём к реализации: создадим первого работника, супервайзера и соединим их в граф.


## Тема 2. Проектирование multi‑agent системы на LangGraph

Теперь, когда мы поняли, зачем нужны несколько агентов и какие паттерны существуют, перейдём к практике. Мы спроектируем систему из трёх специализированных агентов, которые будут работать в цикле, уточняя и дополняя друг друга. Для этого мы используем **LangGraph** — он идеально подходит для multi‑agent, потому что граф позволяет легко задавать переходы между разными узлами, где каждый узел — это отдельный агент.

### 2.1. Три агента: исследователь, критик, писатель

Мы определим трёх агентов, каждый со своей ролью и доступными инструментами:

| Агент | Роль | Инструменты |
|-------|------|-------------|
| **`researcher`** | Ищет информацию в документах и в интернете, собирает факты | `search_docs`, `web_search` |
| **`critic`** | Проверяет найденную информацию на достоверность, логику и полноту | Отсутствуют (использует только LLM) |
| **`writer`** | Пишет финальный ответ на основе материалов от `researcher` и замечаний `critic` | Отсутствуют (использует только LLM) |

### 2.2. Как они будут общаться

Процесс работы выглядит так:

1. **`researcher`** получает вопрос пользователя и ищет информацию, возвращая список фактов.
2. **`critic`** анализирует эти факты: проверяет, достаточно ли их, нет ли противоречий, всё ли логично.
3. **Если `critic` находит ошибки или нехватку данных** — он возвращает запрос на уточнение, и процесс возвращается к `researcher` для дополнительного поиска.
4. **Если `critic` доволен** — управление передаётся `writer`, который пишет финальный ответ.
5. **`writer`** создаёт структурированный, грамотный текст на основе фактов и замечаний критика.

Таким образом, мы получаем **итеративный цикл**: исследователь → критик → (если нужно) исследователь → … → писатель.

### 2.3. Состояние (State)

В LangGraph состояние — это словарь (или TypedDict), который путешествует между узлами. В нашей multi‑agent системе состояние будет содержать:

```python
class MultiAgentState(TypedDict):
    question: str                # исходный вопрос пользователя
    messages: List[BaseMessage]  # история сообщений (для LLM)
    research_result: str         # факты, собранные исследователем
    critic_feedback: str         # замечания критика
    is_ready: bool               # достаточно ли данных для ответа
    final_answer: str            # финальный ответ (заполняется writer)
```

Флаги `is_ready` и наличие `critic_feedback` будут определять, нужно ли возвращаться к исследователю или можно переходить к писателю.

### 2.4. Рёбра графа и логика переходов

Граф состоит из четырёх узлов: три агента и специальный узел-супервайзер, который решает, куда идти дальше. Однако мы можем обойтись без отдельного супервайзера, если встроим логику перехода в функцию после `critic`. В нашем случае:

- **Начало**: узел `researcher`.
- После `researcher` → переходим к `critic`.
- После `critic` → вызываем функцию **`should_continue`**, которая:
  - Если `critic` считает, что данных достаточно (`is_ready == True`) → переход к `writer`.
  - Если `critic` требует уточнений (`is_ready == False`) → возврат к `researcher` (цикл).
- После `writer` → завершение (END).

Таким образом, мы создаём цикл «research → critic → (research) → … → writer», который прерывается только когда критик удовлетворён.

### 2.5. Реализация каждого агента как отдельной функции (узла)

Каждый агент — это функция, которая получает текущее состояние, вызывает LLM с соответствующим системным промптом (и, возможно, инструментами), и возвращает обновлённое состояние.

**Пример структуры узлов:**

```python
def researcher_node(state: MultiAgentState) -> dict:
    # берёт question из state
    # вызывает LLM с инструментами поиска
    # сохраняет результат в research_result
    return {"research_result": "...", "messages": [...]}

def critic_node(state: MultiAgentState) -> dict:
    # анализирует research_result
    # если не хватает данных — устанавливает is_ready = False и пишет feedback
    # если достаточно — is_ready = True
    return {"is_ready": True/False, "critic_feedback": "...", "messages": [...]}

def writer_node(state: MultiAgentState) -> dict:
    # берёт research_result и critic_feedback
    # генерирует финальный ответ
    return {"final_answer": "...", "messages": [...]}
```

### 2.6. Сборка графа в LangGraph

Теперь мы можем собрать граф с помощью `StateGraph`. Мы добавим условное ребро после `critic`, чтобы реализовать цикл.

```python
from langgraph.graph import StateGraph, END

builder = StateGraph(MultiAgentState)

builder.add_node("researcher", researcher_node)
builder.add_node("critic", critic_node)
builder.add_node("writer", writer_node)

builder.set_entry_point("researcher")

# Переход от researcher к critic
builder.add_edge("researcher", "critic")

# Условное ребро от critic
def route_after_critic(state: MultiAgentState) -> str:
    if state.get("is_ready", False):
        return "writer"
    else:
        return "researcher"

builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {
        "writer": "writer",
        "researcher": "researcher"
    }
)

builder.add_edge("writer", END)

graph = builder.compile()
```

**Схема графа словами:**

1. Вход → **Researcher** (исследователь) — собирает факты.
2. Переход к **Critic** (критик) — проверяет факты.
3. После критика:
   - Если факты удовлетворительны → переход к **Writer** (писатель).
   - Если нет → возврат к **Researcher** (дополнительный поиск).
4. **Writer** пишет финальный ответ → завершение.

Этот цикл может повторяться несколько раз, пока критик не примет решение, что информации достаточно.

---

## Что дальше?

В следующей теме мы реализуем каждого агента с реальными инструментами и LLM, а затем протестируем всю систему на практике. Мы увидим, как итеративный процесс улучшает качество ответов и делает систему более надёжной.


## Тема 3. Реализация агентов и их инструментов (скрипт `multi_agent_with_tools.py`)

Теперь, когда у нас есть архитектура и план взаимодействия, мы переходим к реализации. В этой теме мы создадим полноценных агентов с инструментами и свяжем их в единый граф. Мы также учтём важные практические моменты: защиту от пустых ответов, логирование шагов и обработку циклических переходов.

---

### 3.1. Инициализация LLM для каждого агента

Мы будем использовать одну и ту же модель (`qwen2.5:3b`), но с разными системными промптами для каждой роли. Это позволяет каждому агенту «вжиться» в свою роль и выполнять задачу максимально качественно.

```python
from langchain_ollama import ChatOllama

# Базовая модель
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)

# Для исследователя (с инструментами)
llm_researcher = llm.bind_tools([search_docs, web_search])

# Для критика (без инструментов)
llm_critic = llm

# Для писателя (без инструментов)
llm_writer = llm
```

---

### 3.2. Создание инструментов и привязка их только к исследователю

Мы используем три инструмента: поиск в документах, веб-поиск и калькулятор. Только исследователь имеет доступ к поисковым инструментам, а калькулятор доступен всем (хотя в нашем сценарии его использует аналитик, но мы показываем общую архитектуру).

```python
@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний."""
    # В реальном проекте здесь используется retriever из Chroma
    return f"🔍 Результаты поиска по запросу '{query}':\n" + \
           "1. RAG (Retrieval-Augmented Generation) сочетает поиск и генерацию.\n" + \
           "2. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM).\n" + \
           "3. Применяется в чат-ботах, аналитике, научных исследованиях."

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете через DuckDuckGo."""
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if not results:
                return "Ничего не найдено."
            formatted = []
            for i, r in enumerate(results, 1):
                formatted.append(f"{i}. {r['title']}\n   {r['body'][:150]}...\n   {r['href']}")
            return "\n\n".join(formatted)
    except ImportError:
        return "Библиотека duckduckgo-search не установлена. Установите: pip install duckduckgo-search"
    except Exception as e:
        return f"Ошибка веб-поиска: {e}"

@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления."""
    try:
        safe_dict = {'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
                     'pi': math.pi, 'e': math.e, 'abs': abs, 'round': round}
        cleaned = re.sub(r'[^0-9+\-*/%().,sqrt sincostanlogpi eabsround]', '', expression.lower())
        result = eval(cleaned, {"__builtins__": {}}, safe_dict)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

ALL_TOOLS = {
    "search_docs": search_docs,
    "web_search": web_search,
    "calculate": calculate,
}
```

**Инструменты привязываются только к `researcher`** через `bind_tools()`. Критик и писатель не имеют доступа к инструментам, что предотвращает нежелательные вызовы.

---

### 3.3. Функции-узлы: `researcher_node`, `critic_node`, `writer_node`

Каждый узел — это функция, которая принимает текущее состояние и возвращает обновлённое состояние. Мы добавили важные улучшения:

- **Защита от пустых ответов** – если исследователь не смог сформулировать факты, мы подставляем осмысленное сообщение.
- **Логирование длины ответа** – помогает отлаживать, достаточно ли информации получил критик.
- **Чёткая маршрутизация** – критик принимает решение на основе ключевых слов.

#### 3.3.1. Узел `researcher_node` (исследователь)

Исследователь получает вопрос из состояния, вызывает LLM с инструментами, обрабатывает результат и сохраняет факты. Если ответ пустой – подставляется текст-заглушка.

```python
def researcher_node(state: MultiAgentState) -> dict:
    """Агент-исследователь: собирает факты. Гарантирует непустой результат."""
    question = state.get("question", "")
    if not question:
        return {"messages": [AIMessage(content="Нет вопроса для исследования.")]}

    prompt = [SystemMessage(content=RESEARCHER_PROMPT), HumanMessage(content=question)]
    response = llm_researcher.invoke(prompt)

    research_result = None

    # Если были вызовы инструментов – обрабатываем результат
    if hasattr(response, "tool_calls") and response.tool_calls:
        tool_name = response.tool_calls[0]['name']
        tool_args = response.tool_calls[0]['args']
        if tool_name in ALL_TOOLS:
            tool_result = ALL_TOOLS[tool_name].invoke(tool_args)
            # Пытаемся получить осмысленный ответ от LLM на основе результатов поиска
            for attempt in range(2):  # до двух попыток
                final_response = llm_researcher.invoke([
                    SystemMessage(content=RESEARCHER_PROMPT),
                    HumanMessage(content=(
                        f"Вопрос: {question}\n\n"
                        f"Результат поиска:\n{tool_result}\n\n"
                        "Сформулируй факты на основе этого."
                    ))
                ])
                if final_response.content.strip():
                    research_result = final_response.content
                    break
            # Если LLM так и не выдала текст – используем сырой результат поиска как факты
            if not research_result:
                research_result = f"Результаты поиска:\n{tool_result}"
    else:
        research_result = response.content

    # Последняя страховка – абсолютно пустой ответ заменяем осмысленным сообщением
    if not research_result or not research_result.strip():
        research_result = "Информация по данному запросу не найдена."

    print("🔍 Исследователь: собрал факты...")
    return {
        "messages": [AIMessage(content=f"📚 РЕЗУЛЬТАТ ИССЛЕДОВАНИЯ:\n{research_result}")],
        "research_done": True,
        "research_result": research_result,
    }
```

#### 3.3.2. Узел `critic_node` (критик)

Критик анализирует факты и решает, достаточно ли их. Если нет — возвращает запрос на уточнение. Мы добавили логирование длины ответа, чтобы видеть, сколько данных получил критик.

```python
def critic_node(state: MultiAgentState) -> dict:
    """Агент-критик: проверяет факты."""
    research_result = state.get("research_result", "")
    print(f"📊 Критик: получен ответ длиной {len(research_result)} символов")

    if not research_result or not research_result.strip():
        print("📊 Критик: данных нет, возвращаю обратно исследователю.")
        return {
            "messages": [AIMessage(content="Нет данных для проверки.")],
            "is_ready": False,
            "critic_feedback": "Исследование не выполнено. Нужно собрать факты."
        }

    prompt = [SystemMessage(content=CRITIC_PROMPT), HumanMessage(content=f"Факты:\n{research_result}")]
    response = llm.invoke(prompt)
    feedback = response.content

    if any(word in feedback.lower() for word in ["недостаточно", "требуется", "уточнить"]):
        print("📊 Критик: данных недостаточно, требуется уточнение...")
        return {
            "messages": [AIMessage(content=f"📊 ЗАМЕЧАНИЯ КРИТИКА:\n{feedback}")],
            "is_ready": False,
            "critic_feedback": feedback,
        }

    print("📊 Критик: данные удовлетворительны.")
    return {
        "messages": [AIMessage(content=f"📊 КРИТИК ОДОБРЯЕТ:\n{feedback}")],
        "is_ready": True,
        "critic_feedback": feedback,
    }
```

#### 3.3.3. Узел `writer_node` (писатель)

Писатель принимает факты и замечания критика, генерирует финальный ответ. Он не имеет доступа к инструментам, чтобы не отвлекаться на лишние вызовы.

```python
def writer_node(state: MultiAgentState) -> dict:
    """Агент-писатель: пишет финальный ответ."""
    research_result = state.get("research_result", "")
    critic_feedback = state.get("critic_feedback", "")

    context = f"Факты:\n{research_result}\n\nЗамечания критика:\n{critic_feedback}"
    if not context.strip():
        return {"messages": [AIMessage(content="Нет данных для написания ответа.")]}

    prompt = [SystemMessage(content=WRITER_PROMPT), HumanMessage(content=context)]
    response = llm.invoke(prompt)

    print("✍️ Писатель: написал ответ...")
    return {
        "messages": [AIMessage(content=f"📝 ФИНАЛЬНЫЙ ОТВЕТ:\n{response.content}")],
        "writer_done": True,
        "final_answer": response.content,
    }
```

---

### 3.4. Как обновляется состояние

Каждый узел возвращает словарь с полями, которые обновляют состояние. LangGraph автоматически объединяет возвращённые значения с существующим состоянием:

| Узел | Добавляет в состояние |
|------|----------------------|
| `researcher` | `research_done=True`, `research_result`, `messages` |
| `critic` | `is_ready`, `critic_feedback`, `messages` |
| `writer` | `writer_done=True`, `final_answer`, `messages` |

Сообщения от каждого агента добавляются в список `messages` с ролью `AIMessage`. Это позволяет отслеживать историю диалога между агентами.

---

## Полный код `multi_agent_with_tools.py`

Теперь соберём всё вместе в один файл. Код включает все исправления и дополнения, которые мы обсудили.

```python
"""
multi_agent_with_tools.py - Multi‑Agent система с реальными инструментами (исправленная версия)
Лекция 6.6, Тема 3

Архитектура: Исследователь (с поиском) → Критик (проверяет) → Писатель → (цикл, если нужно)
"""

import re
import math
import requests
from datetime import datetime
from typing import TypedDict, List, Annotated, Literal

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# ============================================================================
# 1. ПРОВЕРКА OLLAMA
# ============================================================================

try:
    requests.get("http://localhost:11434/api/tags", timeout=2)
    print("✅ Ollama запущен")
except:
    print("❌ Ollama не запущен! Выполните: ollama serve")
    exit(1)

# ============================================================================
# 2. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний."""
    # В реальном проекте здесь используется retriever из Chroma
    return f"🔍 Результаты поиска по запросу '{query}':\n" + \
           "1. RAG (Retrieval-Augmented Generation) сочетает поиск и генерацию.\n" + \
           "2. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM).\n" + \
           "3. Применяется в чат-ботах, аналитике, научных исследованиях."

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете через DuckDuckGo."""
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if not results:
                return "Ничего не найдено."
            formatted = []
            for i, r in enumerate(results, 1):
                formatted.append(f"{i}. {r['title']}\n   {r['body'][:150]}...\n   {r['href']}")
            return "\n\n".join(formatted)
    except ImportError:
        return "Библиотека duckduckgo-search не установлена. Установите: pip install duckduckgo-search"
    except Exception as e:
        return f"Ошибка веб-поиска: {e}"

@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления."""
    try:
        safe_dict = {'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
                     'pi': math.pi, 'e': math.e, 'abs': abs, 'round': round}
        cleaned = re.sub(r'[^0-9+\-*/%().,sqrt sincostanlogpi eabsround]', '', expression.lower())
        result = eval(cleaned, {"__builtins__": {}}, safe_dict)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

ALL_TOOLS = {
    "search_docs": search_docs,
    "web_search": web_search,
    "calculate": calculate,
}

# ============================================================================
# 3. LLM И СИСТЕМНЫЕ ПРОМПТЫ
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)

RESEARCHER_PROMPT = """
Ты — исследователь. Твоя задача — найти и структурировать информацию по запросу пользователя.
Используй инструменты search_docs (поиск в документах) или web_search (поиск в интернете) для сбора фактов.
Верни список ключевых фактов, цитат или данных. Будь конкретным и полезным.
"""

CRITIC_PROMPT = """
Ты — критик. Твоя задача — проверить предоставленные факты на:
1. Достоверность (не противоречат ли они известным фактам).
2. Полноту (достаточно ли информации для ответа на вопрос).
3. Логику (нет ли внутренних противоречий).

Если фактов достаточно — одобри их и передай писателю.
Если фактов недостаточно или есть ошибки — укажи, что нужно уточнить, и верни запрос исследователю.

Верни структурированный ответ.
"""

WRITER_PROMPT = """
Ты — писатель. Твоя задача — на основе фактов и замечаний критика написать связный, грамотный ответ.
Ответ должен быть понятным, структурированным и полным.
Используй только предоставленные факты, не добавляй свою информацию.
"""

# ============================================================================
# 4. СОСТОЯНИЕ
# ============================================================================

class MultiAgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    research_done: bool
    research_result: str
    is_ready: bool
    critic_feedback: str
    writer_done: bool
    final_answer: str

# ============================================================================
# 5. УЗЛЫ АГЕНТОВ
# ============================================================================

# LLM с инструментами для исследователя
llm_researcher = llm.bind_tools([search_docs, web_search])

def researcher_node(state: MultiAgentState) -> dict:
    """Агент-исследователь: собирает факты. Гарантирует непустой результат."""
    question = state.get("question", "")
    if not question:
        return {"messages": [AIMessage(content="Нет вопроса для исследования.")]}

    prompt = [SystemMessage(content=RESEARCHER_PROMPT), HumanMessage(content=question)]
    response = llm_researcher.invoke(prompt)

    research_result = None

    # Если были вызовы инструментов – обрабатываем результат
    if hasattr(response, "tool_calls") and response.tool_calls:
        tool_name = response.tool_calls[0]['name']
        tool_args = response.tool_calls[0]['args']
        if tool_name in ALL_TOOLS:
            tool_result = ALL_TOOLS[tool_name].invoke(tool_args)
            # Пытаемся получить осмысленный ответ от LLM на основе результатов поиска
            for attempt in range(2):  # до двух попыток
                final_response = llm_researcher.invoke([
                    SystemMessage(content=RESEARCHER_PROMPT),
                    HumanMessage(content=(
                        f"Вопрос: {question}\n\n"
                        f"Результат поиска:\n{tool_result}\n\n"
                        "Сформулируй факты на основе этого."
                    ))
                ])
                if final_response.content.strip():
                    research_result = final_response.content
                    break
            # Если LLM так и не выдала текст – используем сырой результат поиска как факты
            if not research_result:
                research_result = f"Результаты поиска:\n{tool_result}"
    else:
        research_result = response.content

    # Последняя страховка – абсолютно пустой ответ заменяем осмысленным сообщением
    if not research_result or not research_result.strip():
        research_result = "Информация по данному запросу не найдена."

    print("🔍 Исследователь: собрал факты...")
    return {
        "messages": [AIMessage(content=f"📚 РЕЗУЛЬТАТ ИССЛЕДОВАНИЯ:\n{research_result}")],
        "research_done": True,
        "research_result": research_result,
    }

def critic_node(state: MultiAgentState) -> dict:
    """Агент-критик: проверяет факты."""
    research_result = state.get("research_result", "")
    print(f"📊 Критик: получен ответ длиной {len(research_result)} символов")

    if not research_result or not research_result.strip():
        print("📊 Критик: данных нет, возвращаю обратно исследователю.")
        return {
            "messages": [AIMessage(content="Нет данных для проверки.")],
            "is_ready": False,
            "critic_feedback": "Исследование не выполнено. Нужно собрать факты."
        }

    prompt = [SystemMessage(content=CRITIC_PROMPT), HumanMessage(content=f"Факты:\n{research_result}")]
    response = llm.invoke(prompt)
    feedback = response.content

    if any(word in feedback.lower() for word in ["недостаточно", "требуется", "уточнить"]):
        print("📊 Критик: данных недостаточно, требуется уточнение...")
        return {
            "messages": [AIMessage(content=f"📊 ЗАМЕЧАНИЯ КРИТИКА:\n{feedback}")],
            "is_ready": False,
            "critic_feedback": feedback,
        }

    print("📊 Критик: данные удовлетворительны.")
    return {
        "messages": [AIMessage(content=f"📊 КРИТИК ОДОБРЯЕТ:\n{feedback}")],
        "is_ready": True,
        "critic_feedback": feedback,
    }

def writer_node(state: MultiAgentState) -> dict:
    """Агент-писатель: пишет финальный ответ."""
    research_result = state.get("research_result", "")
    critic_feedback = state.get("critic_feedback", "")

    context = f"Факты:\n{research_result}\n\nЗамечания критика:\n{critic_feedback}"
    if not context.strip():
        return {"messages": [AIMessage(content="Нет данных для написания ответа.")]}

    prompt = [SystemMessage(content=WRITER_PROMPT), HumanMessage(content=context)]
    response = llm.invoke(prompt)

    print("✍️ Писатель: написал ответ...")
    return {
        "messages": [AIMessage(content=f"📝 ФИНАЛЬНЫЙ ОТВЕТ:\n{response.content}")],
        "writer_done": True,
        "final_answer": response.content,
    }

# ============================================================================
# 6. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_critic(state: MultiAgentState) -> Literal["writer", "researcher"]:
    """Определяет, куда идти дальше: к писателю или обратно к исследователю."""
    if state.get("is_ready", False):
        return "writer"
    return "researcher"

# ============================================================================
# 7. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(MultiAgentState)

builder.add_node("researcher", researcher_node)
builder.add_node("critic", critic_node)
builder.add_node("writer", writer_node)

builder.set_entry_point("researcher")
builder.add_edge("researcher", "critic")
builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {
        "writer": "writer",
        "researcher": "researcher",
    }
)
builder.add_edge("writer", END)

graph = builder.compile()

# ============================================================================
# 8. ЗАПУСК
# ============================================================================

def run_multi_agent(question: str, recursion_limit: int = 10) -> str:
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "question": question,
        "research_done": False,
        "research_result": "",
        "is_ready": False,
        "critic_feedback": "",
        "writer_done": False,
        "final_answer": "",
    }
    config = {"recursion_limit": recursion_limit}

    try:
        result = graph.invoke(initial_state, config=config)
        for msg in reversed(result["messages"]):
            if hasattr(msg, "content") and "ФИНАЛЬНЫЙ ОТВЕТ" in msg.content:
                return msg.content
        return result["messages"][-1].content
    except Exception as e:
        return f"❌ Ошибка: {e}"

if __name__ == "__main__":
    questions = [
        "Что такое RAG и как он работает?",
        "Сколько будет 25% от 200?",
        "Напиши краткий отчёт о преимуществах локальных LLM.",
    ]

    for q in questions:
        print("\n" + "=" * 60)
        print(f"📝 Вопрос: {q}")
        print("=" * 60)
        answer = run_multi_agent(q, recursion_limit=10)
        print(f"\n✅ Итоговый ответ:\n{answer}")
        print("\n" + "-" * 60)
```

---

## Краткий итог Тема 3

- **Исследователь** (`researcher_node`) — использует инструменты `search_docs` и `web_search` для сбора фактов. Гарантирует непустой результат.
- **Критик** (`critic_node`) — анализирует факты, решает, достаточно ли их, и возвращает обратную связь с логированием.
- **Писатель** (`writer_node`) — генерирует финальный ответ на основе фактов и замечаний.
- **Состояние** хранит всё: вопрос, факты, замечания, флаги готовности.
- **Граф** построен с условным ребром, позволяющим возвращаться к исследователю, если критик не одобрил факты.
- **Исправления** – защита от пустых ответов, обработка ошибок, логирование.

---

**В следующей теме мы протестируем систему на разных вопросах и увидим, как она работает в реальных условиях.**


## Тема 4. Цикл взаимодействия и условия остановки (скрипт `multi_agent_with_tools.py`)

Мы спроектировали агентов и дали им инструменты. Теперь нужно организовать их взаимодействие так, чтобы система не зацикливалась и гарантированно приходила к результату. В этой теме мы разберём, как работает итеративный цикл «исследователь → критик → (если нужно) исследователь → … → писатель», и как мы управляем его остановкой.

---

### 4.1. Как работает цикл

Граф построен вокруг трёх узлов и одного условного ребра:

1. **Точка входа** – узел `researcher`. Он собирает факты и сохраняет их в состояние.
2. **Переход к критику** – после `researcher` управление передаётся узлу `critic`.
3. **Критик** анализирует факты и устанавливает флаг `is_ready`:
   - `is_ready = True` → данные достаточны, можно переходить к писателю.
   - `is_ready = False` → данных недостаточно, критик оставляет замечания и просит уточнить.
4. **Условное ребро** после критика определяет следующий шаг:
   - Если `is_ready == True` → переход к `writer`.
   - Если `is_ready == False` → возврат к `researcher` с замечаниями.
5. **Цикл** повторяется, пока критик не одобрит факты или не будет достигнут лимит итераций.
6. **Писатель** генерирует финальный ответ и завершает работу.

Этот механизм позволяет агенту уточнять информацию, если первоначальных данных недостаточно, и при этом гарантирует завершение.

---

### 4.2. Условия остановки и защита от бесконечного цикла

LangGraph предоставляет встроенный механизм защиты — `recursion_limit`. Если количество шагов графа превышает этот лимит, выполнение прерывается с ошибкой. В нашем коде мы устанавливаем `recursion_limit=10`, что достаточно для нескольких циклов.

Кроме того, в состоянии есть флаг `writer_done`, который не даёт графу вернуться к исследователю после того, как писатель уже отработал.

**Логика остановки в коде:**

```python
def route_after_critic(state: MultiAgentState) -> Literal["writer", "researcher"]:
    if state.get("is_ready", False):
        return "writer"
    return "researcher"
```

Дополнительно, в супервайзере (который у нас встроен в условное ребро) мы не используем LLM, поэтому решение принимается детерминированно и предсказуемо.

---

### 4.3. Пример выполнения с выводом шагов

Запустим систему на вопросе «Что такое RAG и как он работает?». Вот что происходит по шагам:

```
============================================================
📝 Вопрос: Что такое RAG и как он работает?
============================================================
🔍 Исследователь: собрал факты...
📊 Критик: получен ответ длиной 857 символов
📊 Критик: данные удовлетворительны.
✍️ Писатель: написал ответ...

✅ Итоговый ответ:
📝 ФИНАЛЬНЫЙ ОТВЕТ:
Ретривирационно улучшенное генерирование (RAG) — это подход, который используется для повышения производительности искусственного интеллекта...
```

- **Исследователь** нашёл факты (857 символов).
- **Критик** проверил их и одобрил (не нашёл слов «недостаточно», «требуется»).
- **Писатель** создал финальный ответ.

---

### 4.4. Пример с циклом (дополнительный проход)

Для вопроса «Напиши краткий отчёт о преимуществах локальных LLM» критик посчитал данные недостаточными, и система выполнила второй проход:

```
============================================================
📝 Вопрос: Напиши краткий отчёт о преимуществах локальных LLM.
============================================================
🔍 Исследователь: собрал факты...
📊 Критик: получен ответ длиной 743 символов
📊 Критик: данных недостаточно, требуется уточнение...   ← Критик вернул замечания
🔍 Исследователь: собрал факты...                         ← Исследователь уточнил
📊 Критик: получен ответ длиной 822 символов
📊 Критик: данные удовлетворительны.
✍️ Писатель: написал ответ...

✅ Итоговый ответ:
📝 ФИНАЛЬНЫЙ ОТВЕТ:
...
```

Здесь мы видим, как сработал цикл: критик указал на недостаточность данных, исследователь провёл дополнительный поиск, и только после этого ответ был одобрен.

---

### 4.5. Полный код сборки графа (фрагмент)

Граф собирается с помощью `StateGraph`. Условное ребро после критика — это центральный элемент цикла.

```python
builder = StateGraph(MultiAgentState)

builder.add_node("researcher", researcher_node)
builder.add_node("critic", critic_node)
builder.add_node("writer", writer_node)

builder.set_entry_point("researcher")
builder.add_edge("researcher", "critic")
builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {
        "writer": "writer",
        "researcher": "researcher"
    }
)
builder.add_edge("writer", END)

graph = builder.compile()
```

Функция `route_after_critic` читает флаг `is_ready` из состояния и возвращает имя следующего узла.

```python
def route_after_critic(state: MultiAgentState) -> Literal["writer", "researcher"]:
    if state.get("is_ready", False):
        return "writer"
    return "researcher"
```

---

### 4.6. Запуск с защитой от зацикливания

В функции `run_multi_agent` мы передаём `recursion_limit` в конфиг:

```python
def run_multi_agent(question: str, recursion_limit: int = 10) -> str:
    # ...
    config = {"recursion_limit": recursion_limit}
    result = graph.invoke(initial_state, config=config)
    # ...
```

Если количество шагов превысит `recursion_limit`, LangGraph выбросит исключение, и мы вернём сообщение об ошибке.

---

## Краткий итог Тема 4

- **Цикл** организован через условное ребро после критика.
- **Критик** управляет циклом: если факты удовлетворительны → писатель, иначе → исследователь с замечаниями.
- **Остановка** гарантируется флагом `is_ready` и `recursion_limit`.
- **Прозрачность** — каждый шаг логируется в консоль (исследователь, критик, писатель).
- **Граф** собран с использованием `StateGraph`, `add_conditional_edges` и `add_edge`.

---

**В следующей теме мы добавим в систему долгосрочную память и сделаем агентов пригодными для реальных диалогов.**


## Тема 5. Память в multi‑agent и параллелизм (скрипт `multi_agent_system.py`)

Мы построили рабочую multi‑agent систему, в которой исследователь, критик и писатель успешно справляются с разными вопросами – от простых фактов до вычислений и кратких отчётов. Однако в реальных приложениях агентам нужно не только решать текущий вопрос, но и помнить предыдущие обсуждения, а также уметь обрабатывать несколько запросов одновременно. В этой финальной теме мы добавим **память между диалогами** и **параллельную обработку** запросов.

---

### 5.1. Добавление памяти через MemorySaver

В LangGraph память реализуется через **чекпоинтеры** – механизм, который сохраняет полное состояние графа после каждого шага. Это позволяет агентам «помнить» историю диалога в рамках одной сессии.

**Как это работает:**

1. При компиляции графа мы передаём `checkpointer` – объект, который умеет сохранять и загружать состояние.
2. Самый простой чекпоинтер – `MemorySaver` из `langgraph.checkpoint.memory`. Он хранит состояния в оперативной памяти.
3. При вызове графа мы указываем `thread_id` в конфигурации – это идентификатор сессии. Все вызовы с одинаковым `thread_id` будут использовать общую историю.

```python
from langgraph.checkpoint.memory import MemorySaver

# Создаём чекпоинтер
memory = MemorySaver()

# Компилируем граф с чекпоинтером
graph = builder.compile(checkpointer=memory)

# При запуске передаём thread_id
config = {"configurable": {"thread_id": "user_123"}}
result = graph.invoke(initial_state, config=config)
```

**Почему это важно:**  
В диалоговых приложениях пользователь может уточнять вопросы, возвращаться к предыдущим темам или задавать связанные вопросы. Без памяти агент каждый раз начинал бы с чистого листа. С памятью он использует накопленный контекст, что делает общение более естественным и эффективным.

**Для долговременного хранения** можно использовать `SqliteSaver` (установка: `pip install langgraph-checkpoint-sqlite`). Он сохраняет состояние в SQLite, что позволяет не терять историю между перезапусками приложения.

---

### 5.2. Параллелизм в multi‑agent системах

В нашей архитектуре агенты работают последовательно – исследователь → критик → (возможно, повтор) → писатель. Однако бывают ситуации, когда нужно обработать несколько независимых запросов одновременно – например, в чате с несколькими пользователями или при пакетной обработке вопросов.

LangGraph поддерживает **асинхронные методы**: `ainvoke`, `astream`, `astream_events`. Это позволяет запускать несколько экземпляров графа параллельно с помощью `asyncio.gather()`.

**Реализация в классе `MultiAgentSystem`:**

```python
async def arun(self, question: str, thread_id: str = "default") -> str:
    """Асинхронный запуск системы."""
    result = await self.graph.ainvoke(initial_state, config=config)
    return result["messages"][-1].content

async def run_parallel(self, questions: List[str]) -> List[str]:
    """Параллельная обработка нескольких вопросов."""
    tasks = [self.arun(q, thread_id=f"parallel_{i}") for i, q in enumerate(questions)]
    return await asyncio.gather(*tasks)
```

**Преимущества параллелизма:**
- Ускорение обработки пакетных запросов.
- Возможность обслуживать нескольких пользователей одновременно.
- Эффективное использование ресурсов при нескольких независимых задачах.

---

### 5.3. Сборка итогового класса `MultiAgentSystem`

Теперь объединим все улучшения в единый класс. Он включает:

- **Инструменты** – поиск в документах, веб-поиск, калькулятор.
- **Три агента** – исследователь, критик, писатель.
- **Цикл уточнений** – автоматический возврат к исследователю при недостатке данных.
- **Память** – через `MemorySaver` (или `SqliteSaver`).
- **Параллелизм** – асинхронные методы `arun` и `run_parallel`.
- **Защита от бесконечного цикла** – `max_iterations` и `recursion_limit`.

**Полный код `multi_agent_system.py` (готов к запуску):**

```python
"""
multi_agent_system.py - Multi‑Agent система с памятью, параллелизмом и учётом истории
Лекция 6.6, Тема 5 – Финальная версия
"""

import re
import math
import time
import asyncio
import requests
from typing import TypedDict, List, Annotated, Literal, Optional

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Для долговременной памяти (опционально):
# Установите: pip install langgraph-checkpoint-sqlite
# from langgraph.checkpoint.sqlite import SqliteSaver

# ============================================================================
# 1. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний."""
    return f"🔍 Результаты поиска по запросу '{query}':\n" + \
           "1. RAG сочетает поиск и генерацию.\n" + \
           "2. Основные компоненты: ретривер и генератор.\n" + \
           "3. Применяется в чат-ботах, аналитике."

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете через DuckDuckGo (использует ddgs)."""
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if not results:
                return "Ничего не найдено."
            formatted = []
            for i, r in enumerate(results, 1):
                formatted.append(f"{i}. {r['title']}\n   {r['body'][:150]}...\n   {r['href']}")
            return "\n\n".join(formatted)
    except ImportError:
        return "Библиотека ddgs не установлена. Установите: pip install ddgs"
    except Exception as e:
        return f"Ошибка веб-поиска: {e}"

@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления (например, '25% от 200' → '50')."""
    try:
        expr = expression.lower()
        # Простая обработка процентов
        if '%' in expr and 'от' in expr:
            parts = expr.split('от')
            if len(parts) == 2:
                percent = re.sub(r'[^0-9.]', '', parts[0].strip())
                number = re.sub(r'[^0-9.]', '', parts[1].strip())
                if percent and number:
                    return f"Результат: {float(number) * float(percent) / 100}"
        # Обычное вычисление
        safe_dict = {
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
            'pi': math.pi, 'e': math.e, 'abs': abs, 'round': round,
            'log': math.log, 'log10': math.log10
        }
        cleaned = re.sub(r'[^0-9+\-*/%().,sqrt sincostanlogpi eabsround]', '', expr)
        result = eval(cleaned, {"__builtins__": {}}, safe_dict)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

ALL_TOOLS = {
    "search_docs": search_docs,
    "web_search": web_search,
    "calculate": calculate,
}

# ============================================================================
# 2. КЛАСС MULTIAGENTSYSTEM (расширенный)
# ============================================================================

class MultiAgentSystem:
    """
    Система из нескольких агентов с памятью, параллелизмом и учётом истории.
    
    Агенты:
    - researcher: собирает факты с помощью инструментов
    - critic: проверяет факты на достоверность и полноту
    - writer: пишет финальный ответ с учётом истории диалога
    
    Особенности:
    - Память через MemorySaver (или SqliteSaver по желанию)
    - Асинхронные методы для параллельной обработки вопросов
    - Учёт истории диалога в промптах агентов
    - Логирование времени выполнения
    """
    
    def __init__(
        self,
        model_name: str = "qwen2.5:3b",
        temperature: float = 0.0,
        max_predict: int = 512,
        recursion_limit: int = 15,
        max_iterations: int = 3,
        use_sqlite: bool = False
    ):
        """
        Инициализация multi‑agent системы.
        
        Аргументы:
            model_name: имя модели в Ollama
            temperature: температура генерации
            max_predict: максимальное число токенов
            recursion_limit: максимальное число шагов графа
            max_iterations: максимальное число циклов исследователь-критик
            use_sqlite: использовать SqliteSaver вместо MemorySaver (требует установки)
        """
        self.model_name = model_name
        self.temperature = temperature
        self.max_predict = max_predict
        self.recursion_limit = recursion_limit
        self.max_iterations = max_iterations
        self.use_sqlite = use_sqlite

        self.llm = ChatOllama(
            model=model_name,
            temperature=temperature,
            num_predict=max_predict
        )
        self.system_prompts = self._get_system_prompts()
        self.graph = self._build_graph()

    def _get_system_prompts(self) -> dict:
        """Возвращает системные промпты для каждого агента с учётом истории."""
        return {
            "researcher": """
Ты — исследователь. Твоя задача — найти и структурировать информацию по запросу пользователя.
Учитывай историю диалога, чтобы не повторять уже известные факты.
Используй инструменты: search_docs (локальная база), web_search (интернет) или calculate (вычисления).
Если критик дал замечания, учти их и дополни поиск.
Верни список ключевых фактов, цитат или данных. Будь конкретным и полезным.
""",
            "critic": """
Ты — критик. Проверь предоставленные факты на:
1. Достоверность (не противоречат ли известным фактам).
2. Полноту (достаточно ли информации для прямого ответа на вопрос).
3. Логику (нет ли внутренних противоречий).
Учитывай историю диалога, чтобы оценить прогресс.

Если фактов достаточно — одобри и передай писателю.
Если фактов недостаточно или есть ошибки — укажи **конкретно**, чего не хватает.
""",
            "writer": """
Ты — писатель. На основе фактов и истории диалога напиши **краткий, прямой и точный** ответ на вопрос пользователя.
- Если вопрос требует конкретного числового или фактологического ответа — дай его чётко, без лишних пояснений.
- Не повторяй замечания критика в ответе, используй только факты.
- Ответ должен быть связным, но лаконичным.
- Если в истории уже был похожий вопрос, учти это и не повторяйся.
"""
        }

    def _build_graph(self):
        """Строит граф multi‑agent системы с памятью."""
        class MultiAgentState(TypedDict):
            messages: Annotated[List[BaseMessage], add_messages]
            question: str
            research_done: bool
            research_result: str
            is_ready: bool
            critic_feedback: str
            writer_done: bool
            final_answer: str
            iteration: int

        llm_researcher = self.llm.bind_tools([search_docs, web_search, calculate])
        prompts = self.system_prompts
        max_iter = self.max_iterations

        # ---- Узлы ----
        def researcher_node(state: MultiAgentState) -> dict:
            start = time.time()
            question = state.get("question", "")
            if not question:
                return {"messages": [AIMessage(content="Нет вопроса.")]}

            iteration = state.get("iteration", 0) + 1
            critic_feedback = state.get("critic_feedback", "")
            previous_result = state.get("research_result", "")

            # Извлекаем историю диалога (последние 5 сообщений)
            history = state.get("messages", [])
            history_context = "\n".join([m.content for m in history[-5:] if isinstance(m, (HumanMessage, AIMessage))])

            user_content = f"История диалога:\n{history_context}\n\nВопрос: {question}\n"
            if critic_feedback:
                user_content += f"\nЗамечания критика: {critic_feedback}\n"
                user_content += f"Предыдущие факты: {previous_result}\n"
                user_content += "Уточни поиск с учётом этих замечаний."
            else:
                user_content += "Собери факты по этому вопросу."

            response = llm_researcher.invoke([
                SystemMessage(content=prompts["researcher"]),
                HumanMessage(content=user_content)
            ])

            research_result = None
            if hasattr(response, "tool_calls") and response.tool_calls:
                tool_name = response.tool_calls[0]['name']
                tool_args = response.tool_calls[0]['args']
                if tool_name in ALL_TOOLS:
                    tool_result = ALL_TOOLS[tool_name].invoke(tool_args)
                    final_response = llm_researcher.invoke([
                        SystemMessage(content=prompts["researcher"]),
                        HumanMessage(content=(
                            f"Вопрос: {question}\n\n"
                            f"Результат инструмента {tool_name}:\n{tool_result}\n\n"
                            "Сформулируй краткие факты на основе этого."
                        ))
                    ])
                    research_result = final_response.content.strip()
                    if not research_result:
                        research_result = f"Результат: {tool_result}"
            else:
                research_result = response.content.strip()

            if not research_result:
                research_result = "Информация не найдена."

            elapsed = time.time() - start
            print(f"🔍 Исследователь (итерация {iteration}) выполнен за {elapsed:.2f}с")
            return {
                "messages": [AIMessage(content=f"📚 РЕЗУЛЬТАТ ИССЛЕДОВАНИЯ:\n{research_result}")],
                "research_done": True,
                "research_result": research_result,
                "iteration": iteration,
            }

        def critic_node(state: MultiAgentState) -> dict:
            start = time.time()
            research_result = state.get("research_result", "")
            iteration = state.get("iteration", 0)
            print(f"📊 Критик: проверка (итерация {iteration})...")

            if not research_result or not research_result.strip():
                return {
                    "messages": [AIMessage(content="Нет данных для проверки.")],
                    "is_ready": False,
                    "critic_feedback": "Исследование не выполнено. Нужно собрать факты."
                }

            # Добавляем историю в промпт критика
            history = state.get("messages", [])
            history_context = "\n".join([m.content for m in history[-5:] if isinstance(m, (HumanMessage, AIMessage))])

            prompt = [
                SystemMessage(content=prompts["critic"]),
                HumanMessage(content=f"История диалога:\n{history_context}\n\nФакты:\n{research_result}")
            ]
            response = self.llm.invoke(prompt)
            feedback = response.content

            if iteration >= max_iter:
                print("📊 Критик: превышено число итераций, принудительное одобрение.")
                return {
                    "messages": [AIMessage(content=f"📊 КРИТИК ОДОБРЯЕТ (после {iteration} попыток):\n{feedback}")],
                    "is_ready": True,
                    "critic_feedback": feedback,
                }

            negative_words = ["недостаточно", "требуется", "уточнить", "нет", "отсутствует", "проблема"]
            if any(word in feedback.lower() for word in negative_words):
                print("📊 Критик: данных недостаточно, требуется уточнение...")
                return {
                    "messages": [AIMessage(content=f"📊 ЗАМЕЧАНИЯ КРИТИКА:\n{feedback}")],
                    "is_ready": False,
                    "critic_feedback": feedback,
                }

            print("📊 Критик: данные удовлетворительны.")
            elapsed = time.time() - start
            print(f"📊 Критик выполнен за {elapsed:.2f}с")
            return {
                "messages": [AIMessage(content=f"📊 КРИТИК ОДОБРЯЕТ:\n{feedback}")],
                "is_ready": True,
                "critic_feedback": feedback,
            }

        def writer_node(state: MultiAgentState) -> dict:
            start = time.time()
            question = state.get("question", "")
            research_result = state.get("research_result", "")
            # Используем только факты, замечания критика не включаем
            history = state.get("messages", [])
            history_context = "\n".join([m.content for m in history[-5:] if isinstance(m, (HumanMessage, AIMessage))])

            context = f"История диалога:\n{history_context}\n\nВопрос пользователя: {question}\n\nФакты:\n{research_result}"
            if not context.strip():
                return {"messages": [AIMessage(content="Нет данных для написания.")]}

            prompt = [SystemMessage(content=prompts["writer"]), HumanMessage(content=context)]
            response = self.llm.invoke(prompt)
            elapsed = time.time() - start
            print(f"✍️ Писатель выполнен за {elapsed:.2f}с")
            return {
                "messages": [AIMessage(content=f"📝 ФИНАЛЬНЫЙ ОТВЕТ:\n{response.content}")],
                "writer_done": True,
                "final_answer": response.content,
            }

        # ---- Маршрутизация ----
        def route_after_critic(state: MultiAgentState) -> Literal["writer", "researcher"]:
            return "writer" if state.get("is_ready", False) else "researcher"

        # ---- Сборка графа ----
        builder = StateGraph(MultiAgentState)
        builder.add_node("researcher", researcher_node)
        builder.add_node("critic", critic_node)
        builder.add_node("writer", writer_node)

        builder.set_entry_point("researcher")
        builder.add_edge("researcher", "critic")
        builder.add_conditional_edges(
            "critic",
            route_after_critic,
            {"writer": "writer", "researcher": "researcher"}
        )
        builder.add_edge("writer", END)

        # Выбор чекпоинтера
        if self.use_sqlite:
            try:
                from langgraph.checkpoint.sqlite import SqliteSaver
                memory = SqliteSaver.from_conn_string("checkpoints.db")
                print("✅ Используется SqliteSaver (долговременная память)")
            except ImportError:
                print("⚠️ SqliteSaver не установлен. Используется MemorySaver.")
                print("   Установите: pip install langgraph-checkpoint-sqlite")
                memory = MemorySaver()
        else:
            memory = MemorySaver()

        return builder.compile(checkpointer=memory)

    # ---- Синхронный запуск ----
    def run(
        self,
        question: str,
        thread_id: str = "default",
        verbose: bool = True
    ) -> str:
        """
        Запускает multi‑agent систему с заданным вопросом (синхронно).
        
        Аргументы:
            question: вопрос пользователя
            thread_id: идентификатор сессии (для памяти)
            verbose: выводить ли шаги выполнения
        
        Возвращает:
            Финальный ответ агента
        """
        if verbose:
            print("\n" + "=" * 60)
            print(f"📝 Вопрос: {question}")
            print("=" * 60)

        initial_state = {
            "messages": [HumanMessage(content=question)],
            "question": question,
            "research_done": False,
            "research_result": "",
            "is_ready": False,
            "critic_feedback": "",
            "writer_done": False,
            "final_answer": "",
            "iteration": 0,
        }

        config = {
            "configurable": {"thread_id": thread_id},
            "recursion_limit": self.recursion_limit
        }

        try:
            result = self.graph.invoke(initial_state, config=config)
            for msg in reversed(result["messages"]):
                if hasattr(msg, "content") and "ФИНАЛЬНЫЙ ОТВЕТ" in msg.content:
                    return msg.content
            return result["messages"][-1].content
        except Exception as e:
            error_msg = f"❌ Ошибка: {e}"
            if verbose:
                print(error_msg)
            return error_msg

    # ---- Асинхронный запуск (для параллелизма) ----
    async def arun(
        self,
        question: str,
        thread_id: str = "default",
        verbose: bool = True
    ) -> str:
        """
        Асинхронный запуск системы (поддержка параллелизма).
        """
        if verbose:
            print("\n" + "=" * 60)
            print(f"📝 Вопрос: {question}")
            print("=" * 60)

        initial_state = {
            "messages": [HumanMessage(content=question)],
            "question": question,
            "research_done": False,
            "research_result": "",
            "is_ready": False,
            "critic_feedback": "",
            "writer_done": False,
            "final_answer": "",
            "iteration": 0,
        }

        config = {
            "configurable": {"thread_id": thread_id},
            "recursion_limit": self.recursion_limit
        }

        try:
            result = await self.graph.ainvoke(initial_state, config=config)
            for msg in reversed(result["messages"]):
                if hasattr(msg, "content") and "ФИНАЛЬНЫЙ ОТВЕТ" in msg.content:
                    return msg.content
            return result["messages"][-1].content
        except Exception as e:
            error_msg = f"❌ Ошибка: {e}"
            if verbose:
                print(error_msg)
            return error_msg

    async def run_parallel(
        self,
        questions: List[str],
        thread_id_prefix: str = "parallel",
        verbose: bool = True
    ) -> List[str]:
        """
        Параллельная обработка нескольких вопросов.
        
        Аргументы:
            questions: список вопросов
            thread_id_prefix: префикс для идентификаторов сессий
            verbose: выводить ли шаги выполнения
        
        Возвращает:
            Список финальных ответов в том же порядке
        """
        tasks = [
            self.arun(q, thread_id=f"{thread_id_prefix}_{i}", verbose=verbose)
            for i, q in enumerate(questions)
        ]
        return await asyncio.gather(*tasks)

    # ---- Потоковый запуск ----
    def stream(self, question: str, thread_id: str = "default"):
        """
        Потоковый запуск multi‑agent системы.
        
        Генерирует:
            События выполнения (токены, шаги, ошибки)
        """
        initial_state = {
            "messages": [HumanMessage(content=question)],
            "question": question,
            "research_done": False,
            "research_result": "",
            "is_ready": False,
            "critic_feedback": "",
            "writer_done": False,
            "final_answer": "",
            "iteration": 0,
        }
        config = {
            "configurable": {"thread_id": thread_id},
            "recursion_limit": self.recursion_limit
        }
        try:
            for event in self.graph.stream(initial_state, config=config):
                yield event
        except Exception as e:
            yield {"error": str(e)}


# ============================================================================
# 3. ЗАПУСК
# ============================================================================

if __name__ == "__main__":
    # Проверка Ollama
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("✅ Ollama запущен")
    except:
        print("❌ Ollama не запущен! Выполните: ollama serve")
        exit(1)

    # Создаём систему с MemorySaver (можно включить SqliteSaver, установив use_sqlite=True)
    system = MultiAgentSystem(max_iterations=3, use_sqlite=False)

    # Синхронный тест
    questions = [
        "Что такое RAG и как он работает?",
        "Сколько будет 25% от 200?",
        "Напиши краткий отчёт о преимуществах локальных LLM."
    ]
    for q in questions:
        answer = system.run(q, thread_id="user_123")
        print(f"\n✅ Итоговый ответ:\n{answer}")
        print("\n" + "-" * 60)

    # Демонстрация параллельной обработки (асинхронно)
    print("\n🚀 Демонстрация параллельной обработки:")
    async def demo_parallel():
        parallel_questions = [
            "Какие основные компоненты RAG?",
            "Чем локальные LLM лучше облачных?"
        ]
        results = await system.run_parallel(parallel_questions, thread_id_prefix="parallel_demo")
        for i, res in enumerate(results):
            print(f"Ответ {i+1}:\n{res}\n")

    asyncio.run(demo_parallel())
```

---

### 5.4. Результаты тестирования

Тестовый запуск показывает, что система успешно обрабатывает вопросы, при необходимости выполняет циклы уточнений и выдаёт точные ответы. Благодаря `MemorySaver`, каждый новый вопрос в рамках одной сессии использует накопленную историю, а параллельный запуск демонстрирует стабильную работу с несколькими запросами одновременно.

---

## Заключение Лекции 6.6

Поздравляю! Мы построили **полноценную multi‑agent систему**, которая:

1. **Делит задачи** между исследователем, критиком и писателем.
2. **Использует реальные инструменты** – поиск в документах, веб‑поиск, калькулятор.
3. **Работает в цикле**, уточняя информацию при необходимости.
4. **Помнит историю** диалога через `MemorySaver` (или `SqliteSaver` для долговременного хранения).
5. **Может обрабатывать несколько запросов параллельно** через асинхронные методы.
6. **Защищена от бесконечного цикла** механизмами `max_iterations` и `recursion_limit`.

### Что мы научились делать

- Проектировать роли и взаимодействие между агентами.
- Привязывать разные инструменты к разным агентам.
- Строить граф с условным переходом (цикл).
- Добавлять память через чекпоинтеры.
- Использовать асинхронность для параллельной работы.

### Сравнение с одиночным агентом

| Аспект | Одиночный агент | Multi‑agent система |
|--------|-----------------|---------------------|
| **Специализация** | Один на всё | Каждый делает своё |
| **Качество** | Среднее | Высокое (проверка критиком) |
| **Прозрачность** | Трудно отлаживать | Каждый шаг виден |
| **Память** | Есть (через MemorySaver) | Есть (MemorySaver/SqliteSaver) |
| **Параллелизм** | Нет | Да (async/await) |

### Что дальше?

Теперь вы можете:

- **Добавлять новых агентов** – редактора, фактчекера, планировщика.
- **Подключать реальные данные** – заменить заглушки на Chroma, реальный веб-поиск.
- **Интегрировать с интерфейсами** – Streamlit, Telegram, FastAPI.
- **Экспериментировать с самооценкой** – как в Лекции 6.5.

---

**Вы создали систему, которая не просто отвечает, а думает, проверяет и улучшает себя. Это шаг к настоящему искусственному интеллекту, способному работать с неопределённостью и сложностью.**

